<a href="https://colab.research.google.com/github/LinaGarcia1/ProyectoGrado_MCIC-UD/blob/main/Codigos/Cauca/Interpolaciones_kriging_Cauca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1) Normalizar claves en ambos DataFrames
CLAVES = ["Código_SIMMA", "Fecha_even"]

for df in (evt, fechas):
    # ID como texto limpio
    df["Código_SIMMA"] = df["Código_SIMMA"].astype(str).str.strip()
    # Fecha a datetime (formato AAAA-MM-DD). Si tu fecha NO está así, quita "format=..."
    df["Fecha_even"] = (
        df["Fecha_even"].astype(str).str.strip()
        .str.replace(r"[^\d\-]", "-", regex=True)        # homogeniza separadores
    )
    df["Fecha_even"] = pd.to_datetime(df["Fecha_even"], format="%Y-%m-%d", errors="coerce")

# 2) Diagnóstico básico
print("Tipos evt:\n",    evt[CLAVES].dtypes)
print("Tipos fechas:\n", fechas[CLAVES].dtypes)
print("NaT en evt:",    evt["Fecha_even"].isna().sum())
print("NaT en fechas:", fechas["Fecha_even"].isna().sum())

Tipos evt:
 Código_SIMMA            object
Fecha_even      datetime64[ns]
dtype: object
Tipos fechas:
 Código_SIMMA            object
Fecha_even      datetime64[ns]
dtype: object
NaT en evt: 0
NaT en fechas: 0


#**FASE I: INTERPOLACÓN PRECIPITACIÓN KRIGING**

In [ ]:
# Conexión con Google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


En este bloque se buscó generar un listado con las fechas (calendario único de días) a partir de las fechas de cada evento (t₀),  que incluye t₀ y todos los días previos necesarios para cubrir las ventanas temporales definidas (1, 3, 5, 7, 9, 15, 30 y 180 días antes de cada evento).
Ese calendario se guarda en un CSV y sirve como lista mínima de fechas a procesar para posteriormente calcular acumulados de precipitaciones sin repetir ni omitir días relevantes.

In [ ]:
# Dependencias (Colab)
!pip -q install geopandas pyogrio shapely pyproj rtree rasterio pykrige python-dateutil

import os, numpy as np, pandas as pd, geopandas as gpd
from datetime import timedelta
from dateutil.relativedelta import relativedelta

# === Rutas almacenamiento de eventos en Google drive
EVENTS_CSV     = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/dfeventos.csv"
COL_ID_EVT     = "Código_SIMMA"
COL_FECHA_EVT  = "Fecha_even"
COL_LON_EVT    = "Longitud__"
COL_LAT_EVT    = "Latitud__i"
CRS_TRABAJO    = 9377

# Lista Ventanas Excluentes (dias antes del evento)
VENTANAS = [1, 3, 5, 7, 9, 15, 30, 180]

# Cargar eventos y proyectar de crs de WGS84 - Origen Nacional
df_evt = pd.read_csv(EVENTS_CSV, dtype={COL_ID_EVT:str})
df_evt[COL_FECHA_EVT] = pd.to_datetime(df_evt[COL_FECHA_EVT], errors="coerce")
gdf_evt = gpd.GeoDataFrame(df_evt, geometry=gpd.points_from_xy(df_evt[COL_LON_EVT], df_evt[COL_LAT_EVT], crs="EPSG:4326")).to_crs(CRS_TRABAJO)

# 1) Fechas de evento (t0) como date únicas
t0s = pd.to_datetime(gdf_evt[COL_FECHA_EVT], errors="coerce").dropna().dt.date.unique()

# 2) Profundidad máxima que necesitas antes del evento
W_MAX = max(VENTANAS)

# 3) Construir el set: t0 y los W_MAX días previos
fechas = set(t0s)
for t0 in t0s:
    for d in range(1, W_MAX + 1):
        fechas.add(t0 - timedelta(days=d))

# 4) Exportar ordenado
fechas_necesarias = pd.Series(sorted(fechas), name="fecha")
cal_csv = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/calendario_fechas_necesarias.csv"
fechas_necesarias.to_csv(cal_csv, index=False)
print("Calendario de fechas guardado en:", cal_csv, "Nº fechas:", len(fechas_necesarias))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.6/507.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.6 MB/s eta 0:00:00
Calendario de fechas guardado en: /content/drive/MyDrive/TESIS/salida/INTERPOLACION/calendario_fechas_necesarias.csv Nº fechas: 7115


En este módulo se definen las librerías y rutas necesarias para generar interpolaciones de precipitación diaria mediante kriging ordinario, configuradas con parámetros equivalentes a los de ArcGIS Pro para obtener resultados comparables; además, utilizando un flujo escalable con el fin de reducir tiempos de procesamientos.

In [ ]:
# ============================================
# 1) Imports librerias y definicion de rutas
# ===========================================
import os, numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
from pykrige.ok import OrdinaryKriging
import rasterio
from rasterio.transform import from_origin
from datetime import timedelta

# --- Rutas de insumos y resultados Google Drive
RUTA_ESTACIONES   = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/gjoindias.csv"
RUTA_MASCARA      = "/content/drive/MyDrive/TESIS /CAUCA/CAUCA.shp"
CALENDARIO_CSV    = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/calendario_fechas_necesarias.csv"
OUT_RASTERS_DIR   = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosprueba/rasters_diarios_mask"

# --- CRS de trabajo
CRS_TRABAJO = 9377

# --- Columnas en datos ---
COL_FECHA_EST = "Fecha"
COL_P_MM      = "Precipitacion"
COL_GEOM_WKT  = "geometry"    # WKT 'POINT(lon lat)' en gjoindias.csv

COL_ID_EVT    = "Código_SIMMA"
COL_FECHA_EVT = "Fecha_even"
COL_LON_EVT   = "Longitud__"
COL_LAT_EVT   = "Latitud__i"

# --- Parámetros de kriging
CELL   = 500                 # tamaño de celda (m)
VARIO  = "spherical"            # 'spherical' | 'exponential' | 'gaussian'
NUGGET_FRACCION_DEL_SILL = 1e-5 # fracción del sill (bajo -> honra picos)
RANGE_ESCALA = 1.45             # factor sobre un rango sugerido
N_MIN_PTS = 6                   # mínimo de estaciones en el día
SKIP_IF_EXISTS = True           # reanudo: si existe el TIFF, no recalcula


Se carga la máscara territorial del departamento de Cauca y las estaciones con el valor de la precipitación; además se crea una rejilla de puntos dentro de la máscara, que representa los lugares donde se estimará la precipitación mediante el método de kriging.

In [ ]:
# ==============================
# 2) Carga de datos y rejilla
# ==============================
# Máscara
gdf_masc = gpd.read_file(RUTA_MASCARA).to_crs(CRS_TRABAJO)
mask_poly = unary_union(gdf_masc.geometry)

# Estaciones (desde WKT EPSG:4326 -> CRS_TRABAJO)
from shapely import wkt
df_est = pd.read_csv(RUTA_ESTACIONES)
df_est[COL_FECHA_EST] = pd.to_datetime(df_est[COL_FECHA_EST], errors="coerce")
df_est[COL_P_MM]      = pd.to_numeric(df_est[COL_P_MM], errors="coerce")
geom_est = gpd.GeoSeries.from_wkt(df_est[COL_GEOM_WKT])
gdf_est  = gpd.GeoDataFrame(df_est.drop(columns=[COL_GEOM_WKT]), geometry=geom_est, crs="EPSG:4326").to_crs(CRS_TRABAJO)
gdf_est  = gdf_est.dropna(subset=[COL_FECHA_EST, COL_P_MM])

# Rejilla limitada a la máscara (Y descendente => ráster no queda invertido)
minx, miny, maxx, maxy = gdf_masc.total_bounds
nx = int(np.floor((maxx - minx)/CELL))
ny = int(np.floor((maxy - miny)/CELL))
xs = minx + CELL/2.0 + CELL*np.arange(nx)       # izquierda -> derecha
ys = maxy - CELL/2.0 - CELL*np.arange(ny)       # ARRIBA -> ABAJO

grid_points, grid_idx = [], []
for iy, yy in enumerate(ys):
    for ix, xx in enumerate(xs):
        p = Point(xx, yy)
        if mask_poly.contains(p):
            grid_points.append(p)
            grid_idx.append((iy, ix))
pred_x = np.array([p.x for p in grid_points], dtype="float64")
pred_y = np.array([p.y for p in grid_points], dtype="float64")


Se limitan o filtran los eventos de tal manera que solo se tomen aquellos eventos para los que hay información de precipitación.

In [ ]:
# ==============================
# 3) Cargar CALENDARIO de fechas
# ==============================
# Debe tener una columna 'fecha' (YYYY-MM-DD)
if not os.path.exists(CALENDARIO_CSV):
    VENTANAS = [1,3,5,7,9,15,30,180]

    df_evt = pd.read_csv(RUTA_EVENTOS, dtype={COL_ID_EVT:str})
    df_evt[COL_FECHA_EVT] = pd.to_datetime(df_evt[COL_FECHA_EVT], errors="coerce")
    fechas = set(df_evt[COL_FECHA_EVT].dropna().dt.date.unique())
    for t0 in df_evt[COL_FECHA_EVT].dropna().dt.date.values:
        for w in VENTANAS:
            for d in range(1, w+1):
                fechas.add(t0 - timedelta(days=d))
    cal = pd.DataFrame({"fecha": sorted(fechas)})
    cal.to_csv(CALENDARIO_CSV, index=False)
else:
    cal = pd.read_csv(CALENDARIO_CSV, parse_dates=["fecha"])

# Convertir a lista de date (sin componente hora)
fechas = cal["fecha"].dt.date.tolist()

# Filtrar fechas fuera del rango de estaciones para ahorrar intentos
fmin = gdf_est[COL_FECHA_EST].min().date()
fmax = gdf_est[COL_FECHA_EST].max().date()
fechas = [d for d in fechas if (fmin <= d <= fmax)]

len(fechas), fechas[:5], fechas[-5:]


(7115,
 [datetime.date(2005, 4, 8),
  datetime.date(2005, 4, 9),
  datetime.date(2005, 4, 10),
  datetime.date(2005, 4, 11),
  datetime.date(2005, 4, 12)],
 [datetime.date(2025, 1, 16),
  datetime.date(2025, 1, 17),
  datetime.date(2025, 1, 18),
  datetime.date(2025, 1, 19),
  datetime.date(2025, 1, 20)])

Se busca depurar las estaciones redundantes, estimando una escala espacial, y asegurando una salida para cada día: primero por kriging “estable” y, si es necesario, con IDW como respaldo.

In [ ]:
# Eliminar duplicados espaciales (o “casi” duplicados) de estaciones para un día
def dedup_day_gdf(day_gdf, col_p, decimals=0):
    tmp = day_gdf.copy()
    tmp["x"] = tmp.geometry.x
    tmp["y"] = tmp.geometry.y
    tmp["xr"] = tmp["x"].round(decimals)
    tmp["yr"] = tmp["y"].round(decimals)
    g = tmp.groupby(["xr","yr"], as_index=False)[col_p].mean()
    x = g["xr"].to_numpy(float)
    y = g["yr"].to_numpy(float)
    z = g[col_p].to_numpy(float)
    return x, y, z

# Rango sugerido para el variograma que utiliza kriging = 3 × mediana de la distancia al 3er vecino más cercano
def suggest_range_xy(x, y):
    xy = np.c_[x, y]
    if xy.shape[0] < 4:
        return 20000.0
    D = np.sqrt(((xy[None,:,:]-xy[:,None,:])**2).sum(axis=2))
    D.sort(axis=1)
    d3 = np.median(D[:,3]) if D.shape[1] > 3 else np.median(D[:,-1])
    return float(3.0*d3)

# IDW (fallback) para no perder el día
def idw_predict(x, y, z, px, py, k=12, p=2.0):
    tree = cKDTree(np.c_[x,y])
    dists, idxs = tree.query(np.c_[px,py], k=min(k, len(x)))
    dists = np.atleast_2d(dists); idxs = np.atleast_2d(idxs)
    w = 1.0 / np.maximum(dists, 1e-6)**p
    num = (w * z[idxs]).sum(axis=1)
    den = w.sum(axis=1)
    return (num/den).astype("float32")

# Kriging con reintentos (sube nugget y rango si hay singularidad)
def krige_predict_safe(x, y, z, px, py, variogram_model, base_range, base_nugget, sill):
    if sill < 1e-6:
        return np.full(len(px), float(np.mean(z)), dtype="float32"), {"mode":"const"} #Sucede cuando la mayoria de estaciones reportan el mismo valor.

    nug_scales = [1, 2, 4, 8, 16]
    rng_scales = [1.0, 1.5, 2.0]
    # Se utilizan para que cuando el kriging falle por problemas númericos se vuelva a intertar con otros parametros en el variograma.


    for rs in rng_scales:
        for ns in nug_scales:
            vrng = max(base_range * rs, 1.0)
            vnug = max(base_nugget * ns, 1e-6 * max(sill, 1.0))
            vsill = max(sill, 1e-6)
            try:
                OK = OrdinaryKriging(
                    x, y, z,
                    variogram_model=variogram_model,
                    variogram_parameters={"sill": vsill, "range": vrng, "nugget": vnug},
                    enable_plotting=False
                )
                z_pred, _ = OK.execute("points", px, py)
                return np.asarray(z_pred, dtype="float32"), {"mode":"ok", "sill":vsill, "range":vrng, "nugget":vnug}
            except Exception:
                pass

    # Último recurso: IDW
    return idw_predict(x, y, z, px, py, k=min(12, len(x))), {"mode":"idw"}


La finalidad de este código es generar un ráster interpolado precipitación por cada fecha del calendario, registrando además los parámetros usados. Para cada día: (i) omite el cálculo si el GeoTIFF ya existe (reanuda procesos), (ii) descarta fechas con pocas estaciones, (iii) elimina duplicados espaciales de estaciones, (iv) calcula parámetros iniciales del variograma (sill, rango sugerido, nugget) y ejecuta un kriging seguro con reintentos; si el kriging falla o el campo es casi constante, recurre a un valor constante o IDW como respaldo.

In [ ]:
# ===================================================
# 4) Rango y kriging por fecha usando el calendario
# ===================================================
os.makedirs(OUT_RASTERS_DIR, exist_ok=True)
generados = omitidos = 0
log_rows = []  # Lista que permitira visualizar los parámetros usados por día

for fd in fechas:
    out_tif = os.path.join(OUT_RASTERS_DIR, f"precip_{fd}.tif")
    if SKIP_IF_EXISTS and os.path.exists(out_tif):
        generados += 1
        continue

    day = gdf_est[gdf_est[COL_FECHA_EST].dt.date == fd]
    if len(day) < N_MIN_PTS:
        omitidos += 1
        continue

    # --- Deduplicación por coordenada (evita singularidad) ---
    x, y, z = dedup_day_gdf(day, COL_P_MM, decimals=0)  # 0 ≈ 1 m
    if len(x) < 3:
        omitidos += 1
        continue

    # --- Variograma “ganador” con robustez ---
    sill   = float(np.var(z))
    rng0   = suggest_range_xy(x, y) * float(RANGE_ESCALA)
    nug0   = float(NUGGET_FRACCION_DEL_SILL) * max(sill, 1e-6)

    # --- Intenta Ordinary Kriging con los parámetros base; si SciPy lanza LinAlgError: singular matrix, reintenta subiendo
    # --- nugget y/o range.Si el día es muy plano (sill≈0), devuelve constante (modo "const"). Si todos los intentos fallan,
    # --- usa IDW (modo "idw").
    z_pred, info = krige_predict_safe(
        x, y, z,
        pred_x, pred_y,
        variogram_model=VARIO,
        base_range=rng0,
        base_nugget=nug0,
        sill=sill
    )
    z_pred[z_pred < 0] = 0.0

    # --- Ensamblar al grid (NaN fuera de la máscara) ---
    z_grid = np.full((ny, nx), np.nan, dtype="float32")
    for (iy, ix), v in zip(grid_idx, z_pred):
        z_grid[iy, ix] = v

    # --- Escribir GeoTIFF (orientación correcta) ---
    transform = from_origin(minx, maxy, CELL, CELL)
    with rasterio.open(
        out_tif, "w", driver="GTiff",
        height=ny, width=nx, count=1, dtype="float32",
        crs=gdf_masc.crs, transform=transform,
        nodata=np.nan, compress="DEFLATE"
    ) as dst:
        dst.write(z_grid, 1)

    # Configuración de salida de la lista log
    log_rows.append({
        "fecha": str(fd),
        "n_estaciones": int(len(day)),
        "n_unicas": int(len(x)),
        "sill": sill,
        "range_inicial": rng0,
        "nugget_inicial": nug0,
        "modo": info.get("mode")
    })

    generados += 1

print(f"Rásteres generados: {generados}  |  Días omitidos (pocas estaciones): {omitidos}")

if log_rows:
    df_log = pd.DataFrame(log_rows)
    df_log_path = os.path.join(OUT_RASTERS_DIR, "_log_parametros_por_dia.csv")
    df_log.to_csv(df_log_path, index=False)
    print("Log de parámetros guardado en:", df_log_path)


Rásteres generados: 7115  |  Días omitidos (pocas estaciones): 0


## **FASE II: Acumulación de preciptaciones**

En esta fase se construyen los acumulados excluyentes de precipitación para cada evento (t₀), generando un GeoTIFF por combinación evento–ventana (t₀−1…t₀−w) y registrando, además del valor acumulado en la ubicación del evento, la cobertura efectiva de días disponibles en cada ventana; el procedimiento reutiliza rásteres diarios existentes y permite reanudar el proceso para optimizar tiempo de cómputo y asegurar trazabilidad de los resultados.

En este caso dada la cantidad de recursos de procesamiento y de datos fue necesario particionar el procesamiento en tres partes.

In [ ]:
# ==============================
# Setup (constantes, helpers y reanudación)
# ==============================
import os
import shutil
import numpy as np
import pandas as pd
import rasterio
from datetime import timedelta
from rasterio.transform import rowcol
from tqdm.auto import tqdm

# --- Rutas de salida (ajuste si hace falta) ---
OUT_RASTERS_DIR = OUT_RASTERS_DIR if 'OUT_RASTERS_DIR' in globals() else "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosprueba/rasters_diarios_mask"

OUT_ACCUM_DIR = OUT_ACCUM_DIR if 'OUT_ACCUM_DIR' in globals() else "/content/drive/MyDrive/TESIS/PRUEBA/rasters_acumulados1"
os.makedirs(OUT_ACCUM_DIR, exist_ok=True)

# --- Columnas clave en gdf_evt (ya existente en su entorno) ---
ID_COL   = "Código_SIMMA"
DATE_COL = "Fecha_even"

# --- Ventanas excluyentes y banderas ---
VENTANAS = [1, 3, 5, 7, 9, 15, 30, 180]
SAVE_RASTERS_ACCUM = True   # guardar GeoTIFF acumulados por evento/ventana
SAVE_P0_RASTER     = True   # copiar GeoTIFF del día del evento a OUT_ACCUM_DIR
REUSE_EXISTING_ACC = True   # si existe el acumulado TIFF, reutilizarlo
SKIP_IF_FILLED     = True   # si en el CSV ya están todos los valores de un evento, saltarlo

# --- Comprobaciones básicas sobre gdf_evt ---
gdf_evt.columns = gdf_evt.columns.str.strip()
assert ID_COL in gdf_evt.columns, f"Falta la columna {ID_COL} en gdf_evt"
assert DATE_COL in gdf_evt.columns, f"Falta la columna {DATE_COL} en gdf_evt"

if not np.issubdtype(gdf_evt[DATE_COL].dtype, np.datetime64):
    gdf_evt[DATE_COL] = pd.to_datetime(gdf_evt[DATE_COL], errors="coerce", dayfirst=True)

# (opcional) Asegurar CRS consistente con sus GeoTIFF (p. ej., EPSG:9377)
try:
    if gdf_evt.crs is not None:
        if "9377" not in str(gdf_evt.crs):
            gdf_evt = gdf_evt.to_crs(9377)
except Exception:
    pass

# --- Helpers de nombres, fechas y suma de ráster ---
def fname_daily(d):
    """Ruta del ráster diario para la fecha d (tipo date)."""
    return os.path.join(OUT_RASTERS_DIR, f"precip_{d:%Y-%m-%d}.tif")

def dates_window_exclusive(t0d, w):
    """Devuelve [t0-1, ..., t0-w] (ventana excluyente)."""
    return [t0d - timedelta(days=k) for k in range(1, w + 1)]

def _open_first_existing(dates):
    """Abre el primer tif existente para tomar shape/meta base."""
    for d in dates:
        p = fname_daily(d)
        if os.path.exists(p):
            with rasterio.open(p) as src:
                arr = src.read(1).astype("float32")
                meta = src.meta.copy()
            return arr, meta
    return None, None

def sum_rasters(dates_list):
    """Suma de rásteres diarios sobre la misma malla; ignora nodata/NaN."""
    base_arr, base_meta = _open_first_existing(dates_list)
    if base_arr is None:
        return None, None

    H, W = base_arr.shape
    out = np.zeros((H, W), dtype="float32")

    for d in dates_list:
        p = fname_daily(d)
        if not os.path.exists(p):
            continue

        with rasterio.open(p) as src:
            A = src.read(1).astype("float32")
            mask = (~np.isnan(A)) if src.nodata is None else (A != src.nodata)
            out += np.where(mask, A, 0.0).astype("float32")

    return out, base_meta

def sample_array_at_xy(arr, meta, x, y):
    """Extrae el valor de arr en la coordenada (x,y) usando meta['transform']."""
    transform = meta["transform"]
    r, c = rowcol(transform, x, y)

    if (0 <= r < arr.shape[0]) and (0 <= c < arr.shape[1]):
        val = arr[r, c]
        if (meta.get("nodata") is not None) and (val == meta["nodata"]):
            return np.nan
        return float(val)

    return np.nan

# --- CSV de salida (reanudación) ---
REQ_COLS = (
    ["P0_evento"] +
    [f"P{w}_excl" for w in VENTANAS] +
    [f"cov_P{w}_excl" for w in VENTANAS]
)

out_csv = os.path.join(OUT_ACCUM_DIR, "eventos_features_excluyentes2.csv")

if os.path.exists(out_csv):
    df_prev = pd.read_csv(out_csv)
    if DATE_COL in df_prev.columns:
        df_prev[DATE_COL] = pd.to_datetime(df_prev[DATE_COL], errors="coerce").dt.date
    for c in [ID_COL, DATE_COL] + REQ_COLS:
        if c not in df_prev.columns:
            df_prev[c] = np.nan
else:
    df_prev = pd.DataFrame(columns=[ID_COL, DATE_COL] + REQ_COLS)

def get_prev_row(id_val, date_val):
    if df_prev.empty:
        return None

    m = (df_prev[ID_COL] == id_val) & (df_prev[DATE_COL] == date_val)
    if not m.any():
        return None

    return df_prev.loc[m].iloc[0].copy()

def row_is_complete(row):
    if row is None:
        return False

    vals = [row.get(c) for c in REQ_COLS]
    return np.all(pd.notna(vals))

def upsert_row(rec):
    global df_prev

    key_m = (df_prev[ID_COL] == rec[ID_COL]) & (df_prev[DATE_COL] == rec[DATE_COL])

    if key_m.any():
        idx = df_prev.index[key_m][0]
        for k, v in rec.items():
            if k in [ID_COL, DATE_COL]:
                continue
            if k not in df_prev.columns:
                df_prev[k] = np.nan
            if pd.isna(df_prev.at[idx, k]) and not pd.isna(v):
                df_prev.at[idx, k] = v
    else:
        for k in rec.keys():
            if k not in df_prev.columns:
                df_prev[k] = np.nan
        df_prev = pd.concat([df_prev, pd.DataFrame([rec])], ignore_index=True)

def fname_accum(w, t0d, id_val):
    return os.path.join(OUT_ACCUM_DIR, f"accum_P{w}_excl_t0_{t0d}_{id_val}.tif")

# --- Base única de eventos (para luego particionar) ---
gdf_base = (
    gdf_evt
    .dropna(subset=[DATE_COL])
    .drop_duplicates(subset=[ID_COL, DATE_COL])
    .sort_values(DATE_COL)
    .reset_index(drop=True)
)

print(f"Eventos únicos preparados: {len(gdf_base)}")

# --- Función de ejecución por partición ---
def run_partition(CHUNK_ID: int, TOTAL_CHUNKS: int = 3):
    parts = np.array_split(gdf_base, TOTAL_CHUNKS)

    if not (0 <= CHUNK_ID < len(parts)):
        raise ValueError(
            f"CHUNK_ID={CHUNK_ID} fuera de rango. Debe estar entre 0 y {len(parts)-1}."
        )

    gdf_run = parts[CHUNK_ID]
    print(f"Procesando partición {CHUNK_ID+1}/{TOTAL_CHUNKS} con {len(gdf_run)} eventos.")

    for _, ev in tqdm(
        gdf_run.iterrows(),
        total=len(gdf_run),
        desc=f"Eventos chunk {CHUNK_ID+1}",
        unit="evt"
    ):
        if pd.isna(ev[DATE_COL]):
            continue

        t0d = ev[DATE_COL].date()
        idv = ev[ID_COL]
        xev, yev = ev.geometry.x, ev.geometry.y

        prev = get_prev_row(idv, t0d)
        if SKIP_IF_FILLED and row_is_complete(prev):
            continue

        rec = {ID_COL: idv, DATE_COL: t0d}

        if prev is not None:
            for c in REQ_COLS:
                rec[c] = prev.get(c, np.nan)

        # ----- P0_evento -----
        if pd.isna(rec.get("P0_evento")):
            p0_path = fname_daily(t0d)
            if os.path.exists(p0_path):
                with rasterio.open(p0_path) as src:
                    rec["P0_evento"] = float(list(src.sample([(xev, yev)]))[0][0])

                if SAVE_P0_RASTER:
                    out_p0 = os.path.join(OUT_ACCUM_DIR, f"P0_evento_t0_{t0d}_{idv}.tif")
                    if not os.path.exists(out_p0):
                        shutil.copyfile(p0_path, out_p0)
            else:
                rec["P0_evento"] = np.nan

        # ----- Ventanas excluyentes -----
        for w in tqdm(VENTANAS, desc=f"Ventanas t0={t0d}", unit="win", leave=False):
            col_val = f"P{w}_excl"
            col_cov = f"cov_P{w}_excl"

            if pd.notna(rec.get(col_val)) and pd.notna(rec.get(col_cov)):
                continue

            out_tif = fname_accum(w, t0d, idv)
            dias = dates_window_exclusive(t0d, w)

            if REUSE_EXISTING_ACC and os.path.exists(out_tif):
                with rasterio.open(out_tif) as src:
                    rec[col_val] = float(list(src.sample([(xev, yev)]))[0][0])

                n_exist = sum(1 for d in dias if os.path.exists(fname_daily(d)))
                rec[col_cov] = n_exist / len(dias)

            else:
                s_arr, meta = sum_rasters(dias)

                if s_arr is None:
                    rec[col_val] = np.nan
                    rec[col_cov] = 0.0
                else:
                    if SAVE_RASTERS_ACCUM:
                        m = meta.copy()
                        m.update(driver="GTiff", compress="DEFLATE", dtype="float32", count=1)
                        with rasterio.open(out_tif, "w", **m) as dst:
                            dst.write(s_arr, 1)

                    rec[col_val] = sample_array_at_xy(s_arr, meta, xev, yev)
                    n_exist = sum(1 for d in dias if os.path.exists(fname_daily(d)))
                    rec[col_cov] = n_exist / len(dias)

        upsert_row(rec)
        df_prev.to_csv(out_csv, index=False)

    print("Guardado (reanudación):", out_csv)

    try:
        from IPython.display import display
        display(df_prev.tail(5))
    except Exception:
        pass

Eventos únicos preparados: 1499


In [ ]:
# ==============================
# Ejecutar partición 1/3
# ==============================
run_partition(CHUNK_ID=0, TOTAL_CHUNKS=3)


NameError: name 'run_partition' is not defined

In [ ]:
# ==============================
# Ejecutar partición 2/3
# ==============================
run_partition(CHUNK_ID=1, TOTAL_CHUNKS=3)


Procesando partición 2/3 con 500 eventos.


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)


Eventos chunk 2:   0%|          | 0/500 [00:00<?, ?evt/s]

Ventanas t0=2014-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

Ventanas t0=2014-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

...

Ventanas t0=2017-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

Ventanas t0=2017-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

Ventanas t0=2017-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

Guardado (reanudación): /content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosdef/eventos_features_excluyentes2.csv


,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,P180_excl,cov_P1_excl,cov_P3_excl,cov_P5_excl,cov_P7_excl,cov_P9_excl,cov_P15_excl,cov_P30_excl,cov_P180_excl
2997,54816,2017-01-01,1.745855,12.367475,24.049725,35.900108,42.941513,53.127121,123.959122,207.348953,857.479858,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2998,49154,2017-01-01,15.524679,18.319395,33.105324,41.189533,71.982346,122.749573,230.661316,381.113892,1265.244629,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2999,49175,2017-01-01,30.936174,8.116226,47.651226,55.790066,85.049118,127.740952,234.308838,446.817474,1421.013672,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3000,55826,2017-01-01,17.668812,6.859654,36.469669,45.418076,75.085564,107.747063,206.345978,354.690857,1295.518311,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3001,54278,2017-01-01,6.348264,1.509322,11.715802,39.387585,42.445889,54.976776,131.527939,203.539749,900.555237,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
# ==============================
# Ejecutar partición 3/3
# ==============================
run_partition(CHUNK_ID=2, TOTAL_CHUNKS=3)

Procesando partición 3/3 con 499 eventos.


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)


Eventos chunk 3:   0%|          | 0/499 [00:00<?, ?evt/s]

Ventanas t0=2017-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

Ventanas t0=2017-01-01:   0%|          | 0/8 [00:00<?, ?win/s]

...

Ventanas t0=2025-01-10:   0%|          | 0/8 [00:00<?, ?win/s]

Ventanas t0=2025-01-12:   0%|          | 0/8 [00:00<?, ?win/s]

Ventanas t0=2025-01-20:   0%|          | 0/8 [00:00<?, ?win/s]

Guardado (reanudación): /content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosdef/eventos_features_excluyentes2.csv


,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,P180_excl,cov_P1_excl,cov_P3_excl,cov_P5_excl,cov_P7_excl,cov_P9_excl,cov_P15_excl,cov_P30_excl,cov_P180_excl
1991,70317,2025-01-07,16.488888,4.140583,15.303377,52.008102,107.245285,137.138016,217.218307,356.407959,1061.444092,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1992,70325,2025-01-09,15.435035,15.724499,31.822720,48.610878,98.324310,131.284256,331.346558,458.364166,1016.660156,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1993,70328,2025-01-10,16.804043,7.470252,140.368500,149.700623,189.797623,223.773621,348.006775,592.004395,1479.298828,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1994,70332,2025-01-12,3.968175,2.115736,8.415928,27.584414,31.182634,48.445854,107.628548,250.860016,1199.707520,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1995,70337,2025-01-20,17.963026,8.754098,20.699234,23.932497,27.904638,33.514751,82.007271,396.743622,1365.166382,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


**Tratamiento de resultados**

Finalmente, se obtiene como resultado un archivo .csv que utiliza como claves primarias el código SIMMA del evento y la fecha de registro del suceso. En dicho archivo se almacena el valor de precipitación correspondiente al lugar del evento para el día del suceso (t₀), el día anterior (t₀−1) y los acumulados de precipitación calculados de manera excluyente para diferentes ventanas temporales que van desde 1 hasta 180 días previos al evento, proporcionando así una base consolidada de información climática para el análisis posterior.

In [ ]:
import pandas as pd
import numpy as np
import os

# === RUTAS (ajuste) ===
CSVINTPRECIP = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosdef/eventos_features_excluyentes2.csv"

# 1) Leer con separador ';', coma decimal y BOM seguro
read_kw = dict(sep=',', encoding='utf-8-sig', decimal=',', low_memory=False)
df1 = pd.read_csv(CSVINTPRECIP, **read_kw)

#
display(df1.head(5))


,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,P180_excl,cov_P1_excl,cov_P3_excl,cov_P5_excl,cov_P7_excl,cov_P9_excl,cov_P15_excl,cov_P30_excl,cov_P180_excl
0,55407,2017-01-01,36.844871520996094,5.874406337738037,37.138309478759766,40.22029113769531,63.695594787597656,115.28115844726562,237.59182739257807,480.2944641113281,1502.070556640625,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,54574,2017-01-01,6.179741859436035,1.0642279386520386,5.5772199630737305,24.6229248046875,24.642332077026367,32.581275939941406,84.21417236328125,129.8890838623047,678.3890380859375,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,54577,2017-01-01,5.944658279418945,0.3775715827941894,5.067152500152588,23.58658218383789,23.81475257873535,32.441280364990234,86.93085479736328,134.53424072265625,660.1618041992188,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,50565,2017-01-01,6.797356605529785,1.8659464120864868,14.81995964050293,34.75508499145508,41.031158447265625,54.929141998291016,145.38003540039062,242.63742065429688,961.0257568359376,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,48975,2017-01-01,16.8498477935791,16.388404846191406,82.23938751220703,113.65393829345705,152.3359832763672,194.26368713378903,295.53582763671875,541.196533203125,1753.438720703125,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
ID_COL, DATE_COL = "Código_SIMMA", "Fecha_even"

# asegurar tipos limpios
df_nodup = df1.drop_duplicates(subset=[ID_COL, DATE_COL], keep="first").reset_index(drop=True)
after = len(df_nodup)

print(f"Duplicados removidos por [{ID_COL}, {DATE_COL}]: {before - after}")
print(f"Registros finales: {after}")


Duplicados removidos por [Código_SIMMA, Fecha_even]: 1503
Registros finales: 1499


In [ ]:
OUT = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosdef/eventos_features_excluyentes_definitivos.csv"
df_nodup.to_csv(OUT, index=False, encoding="utf-8")
print("Guardado:", OUT)


Guardado: /content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosdef/eventos_features_excluyentes_definitivos.csv


In [ ]:
len(df_nodup)

1499

## **FASE III: Asignación de precipitaciones acumuladas a ubicación de deslizamientos de tierra**

referencia espacial precisa a los valores de precipitación previamente calculados. De esta manera, cada registro en la base de datos adquiere una localización específica dentro del territorio de estudio, permitiendo integrar los resultados

In [ ]:
import os, numpy as np, pandas as pd, geopandas as gpd


In [ ]:
rutaevtcsv="/content/drive/MyDrive/TESIS/salida/INTERPOLACION/rasters_acumuladosdef/eventos_features_excluyentes_definitivos.csv"
rutafechascsv= "/content/drive/MyDrive/TESIS/salida/dfeventos1.csv"


In [ ]:
evt=pd.read_csv(rutaevtcsv)
fechas=pd.read_csv(rutafechascsv)


In [ ]:
display(evt)

,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,P180_excl,cov_P1_excl,cov_P3_excl,cov_P5_excl,cov_P7_excl,cov_P9_excl,cov_P15_excl,cov_P30_excl,cov_P180_excl
0,55407,2017-01-01,36.844872,5.874406,37.138309,40.220291,63.695595,115.281158,237.591827,480.294464,1502.070557,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,54574,2017-01-01,6.179742,1.064228,5.577220,24.622925,24.642332,32.581276,84.214172,129.889084,678.389038,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,54577,2017-01-01,5.944658,0.377572,5.067153,23.586582,23.814753,32.441280,86.930855,134.534241,660.161804,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,50565,2017-01-01,6.797357,1.865946,14.819960,34.755085,41.031158,54.929142,145.380035,242.637421,961.025757,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,48975,2017-01-01,16.849848,16.388405,82.239388,113.653938,152.335983,194.263687,295.535828,541.196533,1753.438721,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1494,54816,2017-01-01,1.745855,12.367475,24.049725,35.900108,42.941513,53.127121,123.959122,207.348953,857.479858,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1495,49154,2017-01-01,15.524679,18.319395,33.105324,41.189533,71.982346,122.749573,230.661316,381.113892,1265.244629,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1496,49175,2017-01-01,30.936174,8.116226,47.651226,55.790066,85.049118,127.740952,234.308838,446.817474,1421.013672,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1497,55826,2017-01-01,17.668812,6.859654,36.469669,45.418076,75.085564,107.747063,206.345978,354.690857,1295.518311,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
display(fechas)

,Código_SIMMA,Fecha_even,Latitud__i,Longitud__,Fecha_t0_1d,Fecha_t0_3d,Fecha_t0_5d,Fecha_t0_7d,Fecha_t0_15d,Fecha_t0_30d,Fecha_t0_180d
0,32966,2005-10-05,2.812108,-76.763061,2005-10-04,2005-10-02,2005-09-30,2005-09-28,2005-09-20,2005-09-05,2005-04-08
1,32968,2005-10-12,2.708464,-76.769711,2005-10-11,2005-10-09,2005-10-07,2005-10-05,2005-09-27,2005-09-12,2005-04-15
2,53721,2005-11-01,2.075828,-76.664594,2005-10-31,2005-10-29,2005-10-27,2005-10-25,2005-10-17,2005-10-02,2005-05-05
3,54376,2005-11-01,2.188815,-76.793900,2005-10-31,2005-10-29,2005-10-27,2005-10-25,2005-10-17,2005-10-02,2005-05-05
4,55798,2005-11-01,2.621024,-76.574959,2005-10-31,2005-10-29,2005-10-27,2005-10-25,2005-10-17,2005-10-02,2005-05-05
...,...,...,...,...,...,...,...,...,...,...,...
1512,70317,2025-01-07,2.552000,-76.566000,2025-01-06,2025-01-04,2025-01-02,2024-12-31,2024-12-23,2024-12-08,2024-07-11
1513,70325,2025-01-09,2.132000,-76.776000,2025-01-08,2025-01-06,2025-01-04,2025-01-02,2024-12-25,2024-12-10,2024-07-13
1514,70328,2025-01-10,2.656000,-76.536000,2025-01-09,2025-01-07,2025-01-05,2025-01-03,2024-12-26,2024-12-11,2024-07-14
1515,70332,2025-01-12,1.397000,-76.513000,2025-01-11,2025-01-09,2025-01-07,2025-01-05,2024-12-28,2024-12-13,2024-07-16


In [ ]:
# 1) Normalizar claves en ambos DataFrames
CLAVES = ["Código_SIMMA", "Fecha_even"]

for df in (evt, fechas):
    # ID como texto limpio
    df["Código_SIMMA"] = df["Código_SIMMA"].astype(str).str.strip()
    # Fecha a datetime (formato AAAA-MM-DD). Si tu fecha NO está así, quita "format=..."
    df["Fecha_even"] = (
        df["Fecha_even"].astype(str).str.strip()
        .str.replace(r"[^\d\-]", "-", regex=True)        # homogeniza separadores
    )
    df["Fecha_even"] = pd.to_datetime(df["Fecha_even"], format="%Y-%m-%d", errors="coerce")

# 2) Diagnóstico básico
print("Tipos evt:\n",    evt[CLAVES].dtypes)
print("Tipos fechas:\n", fechas[CLAVES].dtypes)
print("NaT en evt:",    evt["Fecha_even"].isna().sum())
print("NaT en fechas:", fechas["Fecha_even"].isna().sum())

Tipos evt:
 Código_SIMMA            object
Fecha_even      datetime64[ns]
dtype: object
Tipos fechas:
 Código_SIMMA            object
Fecha_even      datetime64[ns]
dtype: object
NaT en evt: 0
NaT en fechas: 0


In [ ]:
# 3) Opcional: eliminar filas sin fecha válida y duplicados por clave
evt_ok    = evt.dropna(subset=["Fecha_even"]).drop_duplicates(subset=CLAVES, keep="first")
fechas_ok = fechas.dropna(subset=["Fecha_even"]).drop_duplicates(subset=CLAVES, keep="first")


In [ ]:

# 4) Merge (inner) y conteo
merged = evt_ok.merge(fechas_ok, on=CLAVES, how="inner", suffixes=("_evt", "_fechas"), indicator=True)
print("Filas unidas (inner):", len(merged))

# 5) (Opcional) Auditoría rápida con outer para ver si algo no cruza
outer = evt_ok.merge(fechas_ok, on=CLAVES, how="outer", indicator=True)
print(outer["_merge"].value_counts())

Filas unidas (inner): 1499
_merge
both          1499
left_only        0
right_only       0
Name: count, dtype: int64


In [ ]:
display(merged)

,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,...,Latitud__i,Longitud__,Fecha_t0_1d,Fecha_t0_3d,Fecha_t0_5d,Fecha_t0_7d,Fecha_t0_15d,Fecha_t0_30d,Fecha_t0_180d,_merge
0,55407,2017-01-01,36.844872,5.874406,37.138309,40.220291,63.695595,115.281158,237.591827,480.294464,...,2.625347,-76.749311,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
1,54574,2017-01-01,6.179742,1.064228,5.577220,24.622925,24.642332,32.581276,84.214172,129.889084,...,2.072427,-76.615758,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
2,54577,2017-01-01,5.944658,0.377572,5.067153,23.586582,23.814753,32.441280,86.930855,134.534241,...,2.079586,-76.630654,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
3,50565,2017-01-01,6.797357,1.865946,14.819960,34.755085,41.031158,54.929142,145.380035,242.637421,...,2.174359,-76.699333,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
4,48975,2017-01-01,16.849848,16.388405,82.239388,113.653938,152.335983,194.263687,295.535828,541.196533,...,2.709654,-76.848312,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1494,54816,2017-01-01,1.745855,12.367475,24.049725,35.900108,42.941513,53.127121,123.959122,207.348953,...,2.293859,-76.563693,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
1495,49154,2017-01-01,15.524679,18.319395,33.105324,41.189533,71.982346,122.749573,230.661316,381.113892,...,2.570650,-76.671417,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
1496,49175,2017-01-01,30.936174,8.116226,47.651226,55.790066,85.049118,127.740952,234.308838,446.817474,...,2.640503,-76.772842,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both
1497,55826,2017-01-01,17.668812,6.859654,36.469669,45.418076,75.085564,107.747063,206.345978,354.690857,...,2.658304,-76.686396,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both


In [ ]:
# =========================
gdf = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(merged[LON_COL], merged[LAT_COL]),
    crs="EPSG:4326"
)


In [ ]:
display(gdf)

,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,...,Longitud__,Fecha_t0_1d,Fecha_t0_3d,Fecha_t0_5d,Fecha_t0_7d,Fecha_t0_15d,Fecha_t0_30d,Fecha_t0_180d,_merge,geometry
0,55407,2017-01-01,36.844872,5.874406,37.138309,40.220291,63.695595,115.281158,237.591827,480.294464,...,-76.749311,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.74931 2.62535)
1,54574,2017-01-01,6.179742,1.064228,5.577220,24.622925,24.642332,32.581276,84.214172,129.889084,...,-76.615758,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.61576 2.07243)
2,54577,2017-01-01,5.944658,0.377572,5.067153,23.586582,23.814753,32.441280,86.930855,134.534241,...,-76.630654,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.63065 2.07959)
3,50565,2017-01-01,6.797357,1.865946,14.819960,34.755085,41.031158,54.929142,145.380035,242.637421,...,-76.699333,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.69933 2.17436)
4,48975,2017-01-01,16.849848,16.388405,82.239388,113.653938,152.335983,194.263687,295.535828,541.196533,...,-76.848312,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.84831 2.70965)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1494,54816,2017-01-01,1.745855,12.367475,24.049725,35.900108,42.941513,53.127121,123.959122,207.348953,...,-76.563693,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.56369 2.29386)
1495,49154,2017-01-01,15.524679,18.319395,33.105324,41.189533,71.982346,122.749573,230.661316,381.113892,...,-76.671417,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.67142 2.57065)
1496,49175,2017-01-01,30.936174,8.116226,47.651226,55.790066,85.049118,127.740952,234.308838,446.817474,...,-76.772842,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.77284 2.6405)
1497,55826,2017-01-01,17.668812,6.859654,36.469669,45.418076,75.085564,107.747063,206.345978,354.690857,...,-76.686396,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,both,POINT (-76.6864 2.6583)


In [ ]:
print(gdf.columns.tolist())


['Código_SIMMA', 'Fecha_even', 'P0_evento', 'P1_excl', 'P3_excl', 'P5_excl', 'P7_excl', 'P9_excl', 'P15_excl', 'P30_excl', 'P180_excl', 'cov_P1_excl', 'cov_P3_excl', 'cov_P5_excl', 'cov_P7_excl', 'cov_P9_excl', 'cov_P15_excl', 'cov_P30_excl', 'cov_P180_excl', 'Latitud__i', 'Longitud__', 'Fecha_t0_1d', 'Fecha_t0_3d', 'Fecha_t0_5d', 'Fecha_t0_7d', 'Fecha_t0_15d', 'Fecha_t0_30d', 'Fecha_t0_180d', '_merge', 'geometry']


In [ ]:
# 1) Lista exacta de columnas que quieres conservar (en ese orden)
cols_deseadas = [
    "Código_SIMMA", "Fecha_even", "P0_evento","P1_excl", "P3_excl", "P5_excl", "P7_excl",
    "P9_excl", "P15_excl", "P30_excl", "P180_excl", "Latitud__i",
    "Longitud__", "Fecha_t0_1d", "Fecha_t0_3d", "Fecha_t0_5d",
    "Fecha_t0_7d", "Fecha_t0_15d", "Fecha_t0_30d", "Fecha_t0_180d",
    "geometry"
]

In [ ]:
out_shp = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/tablas/evt_fechas_con_puntos1.shp"
gdf.to_file(out_shp, driver="ESRI Shapefile", encoding="UTF-8")
print("Guardado:", out_shp)

/tmp/ipython-input-2656503875.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(out_shp, driver="ESRI Shapefile", encoding="UTF-8")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Código_SIMMA' to 'Código_SI'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Field Fecha_even create as date field, though DateTime requested.
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'cov_P1_excl' to 'cov_P1_exc'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'cov_P3_excl' to 'cov_P3_exc'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'cov_P5_excl' to 'cov_P5_exc'
  ogr_write(
/usr/local/lib/python3.12/dist-pack

Guardado: /content/drive/MyDrive/TESIS/salida/INTERPOLACION/tablas/evt_fechas_con_puntos1.shp


In [ ]:
gdf_sel = gdf[cols_deseadas].copy()


In [ ]:
display(gdf_sel)

,Código_SIMMA,Fecha_even,P0_evento,P1_excl,P3_excl,P5_excl,P7_excl,P9_excl,P15_excl,P30_excl,...,Latitud__i,Longitud__,Fecha_t0_1d,Fecha_t0_3d,Fecha_t0_5d,Fecha_t0_7d,Fecha_t0_15d,Fecha_t0_30d,Fecha_t0_180d,geometry
0,55407,2017-01-01,36.844872,5.874406,37.138309,40.220291,63.695595,115.281158,237.591827,480.294464,...,2.625347,-76.749311,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.74931 2.62535)
1,54574,2017-01-01,6.179742,1.064228,5.577220,24.622925,24.642332,32.581276,84.214172,129.889084,...,2.072427,-76.615758,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.61576 2.07243)
2,54577,2017-01-01,5.944658,0.377572,5.067153,23.586582,23.814753,32.441280,86.930855,134.534241,...,2.079586,-76.630654,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.63065 2.07959)
3,50565,2017-01-01,6.797357,1.865946,14.819960,34.755085,41.031158,54.929142,145.380035,242.637421,...,2.174359,-76.699333,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.69933 2.17436)
4,48975,2017-01-01,16.849848,16.388405,82.239388,113.653938,152.335983,194.263687,295.535828,541.196533,...,2.709654,-76.848312,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.84831 2.70965)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1494,54816,2017-01-01,1.745855,12.367475,24.049725,35.900108,42.941513,53.127121,123.959122,207.348953,...,2.293859,-76.563693,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.56369 2.29386)
1495,49154,2017-01-01,15.524679,18.319395,33.105324,41.189533,71.982346,122.749573,230.661316,381.113892,...,2.570650,-76.671417,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.67142 2.57065)
1496,49175,2017-01-01,30.936174,8.116226,47.651226,55.790066,85.049118,127.740952,234.308838,446.817474,...,2.640503,-76.772842,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.77284 2.6405)
1497,55826,2017-01-01,17.668812,6.859654,36.469669,45.418076,75.085564,107.747063,206.345978,354.690857,...,2.658304,-76.686396,2016-12-31,2016-12-29,2016-12-27,2016-12-25,2016-12-17,2016-12-02,2016-07-05,POINT (-76.6864 2.6583)


In [ ]:
print(gdf_sel.columns.tolist())


['Código_SIMMA', 'Fecha_even', 'P0_evento', 'P1_excl', 'P3_excl', 'P5_excl', 'P7_excl', 'P9_excl', 'P15_excl', 'P30_excl', 'P180_excl', 'Latitud__i', 'Longitud__', 'Fecha_t0_1d', 'Fecha_t0_3d', 'Fecha_t0_5d', 'Fecha_t0_7d', 'Fecha_t0_15d', 'Fecha_t0_30d', 'Fecha_t0_180d', 'geometry']


In [ ]:
# Exporta a GeoPackage (mantiene nombres largos y fechas)
gdf_sel.to_file("eventos.gpkg", layer="eventos", driver="GPKG")

In [ ]:
out_shp = "/content/drive/MyDrive/TESIS/salida/INTERPOLACION/evt_precifinal.gpkg"
gdf_sel.to_file(out_shp,  layer="eventos", driver="GPKG")
print("Guardado:", out_shp)

Guardado: /content/drive/MyDrive/TESIS/salida/INTERPOLACION/evt_precifinal.gpkg


In [ ]:
# ============================================================
# 1. CONECTAR GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 2. CORREGIR NOTEBOOK PARA GITHUB SIN BORRAR SALIDAS
# ============================================================

import nbformat
from pathlib import Path
import shutil

# Ruta del notebook original en Drive
ruta_original = Path("/content/drive/MyDrive/TESIS-BOYACA/Implementacion_algoritmos.ipynb")

# Crear una copia corregida para GitHub
ruta_corregida = ruta_original.with_name(ruta_original.stem + "_GH.ipynb")

# Crear copia de seguridad adicional
ruta_backup = ruta_original.with_name(ruta_original.stem + "_BACKUP.ipynb")
shutil.copy(ruta_original, ruta_backup)

# Leer notebook
nb = nbformat.read(ruta_original, as_version=4)

# Eliminar SOLO el metadato problemático
# Esto NO borra salidas, gráficos, tablas ni resultados de celdas
nb.metadata.pop("widgets", None)

# Guardar versión corregida
nbformat.write(nb, ruta_corregida)

print("Notebook corregido correctamente.")
print("Archivo original:", ruta_original)
print("Copia de seguridad:", ruta_backup)
print("Archivo para subir a GitHub:", ruta_corregida)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')